Installing dependencies

In [1]:
!pip install -U langchain langsmith langchain-groq pydantic pandas python-dotenv

  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached pydantic-2.13.0-py3-none-any.whl.metadata (107 kB)
  Using cached pydantic_core-2.46.0-cp313-cp313-win_amd64.whl.metadata (6.7 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached pydantic-2.13.0-py3-none-any.whl (471 kB)
Using cached pydantic_core-2.46.0-cp313-cp313-win_amd64.whl (2.1 MB)
Using cached langchain_groq-1.1.2-py3-none-any.whl (19 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.7 MB 16.6 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.7 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 MB 19.1 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 16.7 MB/s  0:00:00

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.27.2

    Uninstalling py

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.


Environment setup

In [2]:
import os

# --- Groq ---
os.environ["GROQ_API_KEY"] = "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"

# --- LangSmith / Tracing ---
os.environ["LANGCHAIN_TRACING_V2"] = "true"   # assignment requirement
os.environ["LANGSMITH_TRACING"] = "true"      # current LangSmith docs
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = "Resume-Screening-Groq"

print("Environment variables set successfully.")

Environment variables set successfully.


Importing packages

In [3]:
import json
import pandas as pd
from typing import List, Dict, Any

from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

Initialize the Groq model

In [28]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq model initialized.")

Groq model initialized.


Sample Job Description and Resumes

In [5]:
job_description = """
Role: Data Scientist

Required Skills:
- Python
- Machine Learning
- Statistics
- SQL
- Data Visualization
- Model Evaluation

Preferred Tools:
- Pandas
- NumPy
- Scikit-learn
- TensorFlow or PyTorch
- Jupyter Notebook
- Git

Experience:
- 2+ years in data analysis / machine learning
- Experience building predictive models
- Experience cleaning and preparing datasets
- Ability to communicate insights clearly

Responsibilities:
- Build ML models
- Analyze structured data
- Perform feature engineering
- Communicate findings to business teams
"""

strong_resume = """
John Doe
Experience: 3 years as Data Scientist at ABC Analytics.
Worked on customer churn prediction, sales forecasting, and fraud detection.
Skills: Python, Machine Learning, Statistics, SQL, Data Visualization, Feature Engineering, Model Evaluation
Tools: Pandas, NumPy, Scikit-learn, TensorFlow, Jupyter Notebook, Git, Matplotlib
Achievements:
- Improved churn prediction accuracy by 18%
- Built end-to-end ML pipelines
- Presented insights to business stakeholders
"""

average_resume = """
Sara Khan
Experience: 1.5 years as Data Analyst.
Worked on dashboards, reporting, and some machine learning experiments.
Skills: Python, SQL, Data Visualization, Basic Machine Learning
Tools: Pandas, NumPy, Jupyter Notebook, Excel
Achievements:
- Built weekly reporting dashboards
- Cleaned and analyzed customer datasets
- Assisted senior team members in model testing
"""

weak_resume = """
Rohit Sharma
Experience: Fresher.
Looking for an entry-level role in technology.
Skills: MS Office, Communication, HTML, CSS
Tools: Word, PowerPoint
Achievements:
- College presentation projects
- Participated in coding club
"""

Defining structured outputs

In [6]:
class ExtractionOutput(BaseModel):
    skills: List[str] = Field(description="Skills explicitly mentioned in the resume")
    tools: List[str] = Field(description="Tools explicitly mentioned in the resume")
    experience_summary: str = Field(description="Short summary of experience from resume only")
    years_of_experience: str = Field(description="Years of experience if explicitly present, else 'Not explicitly mentioned'")

class MatchOutput(BaseModel):
    matched_skills: List[str]
    missing_skills: List[str]
    matched_tools: List[str]
    missing_tools: List[str]
    experience_match: str
    fit_level: str

class ScoreOutput(BaseModel):
    score: int = Field(description="Score between 0 and 100")
    scoring_reason: str = Field(description="Short reason for score")

class ExplanationOutput(BaseModel):
    explanation: str = Field(description="Detailed explanation of why candidate got the score")
    strengths: List[str]
    gaps: List[str]
    recommendation: str

Create output parsers

In [7]:
extraction_parser = JsonOutputParser(pydantic_object=ExtractionOutput)
match_parser = JsonOutputParser(pydantic_object=MatchOutput)
score_parser = JsonOutputParser(pydantic_object=ScoreOutput)
explanation_parser = JsonOutputParser(pydantic_object=ExplanationOutput)
text_parser = StrOutputParser()

Prompt templates

In [8]:
extraction_prompt = PromptTemplate(
    template="""
You are an AI resume screening assistant.

Your task is to extract only information that is explicitly present in the resume.

Rules:
1. Do NOT assume any skill that is not mentioned.
2. Do NOT infer hidden experience.
3. If years of experience are not explicit, return "Not explicitly mentioned".
4. Keep extracted lists concise and factual.
5. Output must be valid JSON only.

{format_instructions}

Resume:
{resume_text}
""",
    input_variables=["resume_text"],
    partial_variables={"format_instructions": extraction_parser.get_format_instructions()}
)

In [9]:
match_prompt = PromptTemplate(
    template="""
You are an AI recruiter assistant.

Compare the extracted resume information with the job description.

Rules:
1. Use only the extracted resume data and the job description.
2. Do NOT assume missing skills are present.
3. Fit level must be one of: Strong, Average, Weak
4. Output valid JSON only.

{format_instructions}

Job Description:
{job_description}

Extracted Resume Data:
{extracted_data}
""",
    input_variables=["job_description", "extracted_data"],
    partial_variables={"format_instructions": match_parser.get_format_instructions()}
)

In [10]:
score_prompt = PromptTemplate(
    template="""
You are an AI scoring system for resume screening.

Give a score from 0 to 100 based on:
- Skills match
- Tools match
- Experience relevance
- Overall alignment with the role

Suggested scoring logic:
- 85 to 100: Strong fit
- 60 to 84: Average fit
- 0 to 59: Weak fit

Rules:
1. Do not give random scores.
2. Score must reflect the provided match analysis.
3. Output valid JSON only.

{format_instructions}

Match Analysis:
{match_data}
""",
    input_variables=["match_data"],
    partial_variables={"format_instructions": score_parser.get_format_instructions()}
)

In [11]:
explanation_prompt = PromptTemplate(
    template="""
You are an explainable AI recruiter assistant.

Write a clear explanation for the candidate's score.

Rules:
1. Mention strengths.
2. Mention missing requirements.
3. Explain the score in plain English.
4. Give a final recommendation.
5. Use only the resume, match analysis, and score provided.
6. Output valid JSON only.

{format_instructions}

Job Description:
{job_description}

Resume:
{resume_text}

Match Analysis:
{match_data}

Score Data:
{score_data}
""",
    input_variables=["job_description", "resume_text", "match_data", "score_data"],
    partial_variables={"format_instructions": explanation_parser.get_format_instructions()}
)

Build LCEL chains

In [12]:
extraction_chain = extraction_prompt | llm | extraction_parser
matching_chain = match_prompt | llm | match_parser
scoring_chain = score_prompt | llm | score_parser
explanation_chain = explanation_prompt | llm | explanation_parser

Main pipeline function

In [13]:
def evaluate_resume(resume_text: str, job_description: str, candidate_name: str) -> Dict[str, Any]:
    # Step 1: Extract
    extracted = extraction_chain.invoke({"resume_text": resume_text})
    
    # Step 2: Match
    matched = matching_chain.invoke({
        "job_description": job_description,
        "extracted_data": json.dumps(extracted, indent=2)
    })
    
    # Step 3: Score
    scored = scoring_chain.invoke({
        "match_data": json.dumps(matched, indent=2)
    })
    
    # Step 4: Explain
    explained = explanation_chain.invoke({
        "job_description": job_description,
        "resume_text": resume_text,
        "match_data": json.dumps(matched, indent=2),
        "score_data": json.dumps(scored, indent=2)
    })
    
    return {
        "candidate_name": candidate_name,
        "extraction": extracted,
        "match": matched,
        "score": scored,
        "explanation": explained
    }

Run all 3 candidates

In [14]:
strong_result = evaluate_resume(strong_resume, job_description, "Strong Candidate")
average_result = evaluate_resume(average_resume, job_description, "Average Candidate")
weak_result = evaluate_resume(weak_resume, job_description, "Weak Candidate")

Display results neatly

In [15]:
def print_result(result: Dict[str, Any]):
    print("="*80)
    print("Candidate:", result["candidate_name"])
    print("-"*80)
    print("EXTRACTION:")
    print(json.dumps(result["extraction"], indent=2))
    print("-"*80)
    print("MATCH:")
    print(json.dumps(result["match"], indent=2))
    print("-"*80)
    print("SCORE:")
    print(json.dumps(result["score"], indent=2))
    print("-"*80)
    print("EXPLANATION:")
    print(json.dumps(result["explanation"], indent=2))
    print("="*80)

print_result(strong_result)
print_result(average_result)
print_result(weak_result)

Candidate: Strong Candidate
--------------------------------------------------------------------------------
EXTRACTION:
{
  "skills": [
    "Python",
    "Machine Learning",
    "Statistics",
    "SQL",
    "Data Visualization",
    "Feature Engineering",
    "Model Evaluation"
  ],
  "tools": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "Jupyter Notebook",
    "Git",
    "Matplotlib"
  ],
  "experience_summary": "Data Scientist at ABC Analytics",
  "years_of_experience": "3 years"
}
--------------------------------------------------------------------------------
MATCH:
{
  "matched_skills": [
    "Python",
    "Machine Learning",
    "Statistics",
    "SQL",
    "Data Visualization",
    "Model Evaluation"
  ],
  "missing_skills": [],
  "matched_tools": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "Jupyter Notebook",
    "Git"
  ],
  "missing_tools": [],
  "experience_match": "Matched",
  "fit_level": "Strong"
}
--------------------

Summary table

In [18]:
summary_data = [
    {
        "Candidate": strong_result["candidate_name"],
        "Score": strong_result["score"]["score"],
        "Fit Level": strong_result["match"]["fit_level"],
        "Recommendation": strong_result["explanation"]["recommendation"]
    },
    {
        "Candidate": average_result["candidate_name"],
        "Score": average_result["score"]["score"],
        "Fit Level": average_result["match"]["fit_level"],
        "Recommendation": average_result["explanation"]["recommendation"]
    },
    {
        "Candidate": weak_result["candidate_name"],
        "Score": weak_result["score"]["score"],
        "Fit Level": weak_result["match"]["fit_level"],
        "Recommendation": weak_result["explanation"]["recommendation"]
    }
]

for row in summary_data:
    print("\nCandidate:", row["Candidate"])
    print("Score:", row["Score"])
    print("Fit Level:", row["Fit Level"])
    print("Recommendation:", row["Recommendation"])


Candidate: Strong Candidate
Score: 100
Fit Level: Strong
Recommendation: Based on the strong fit and relevant experience, it is highly recommended to move forward with the candidate, John Doe, to the next stage of the hiring process.

Candidate: Average Candidate
Score: 70
Fit Level: Average
Recommendation: The candidate shows potential but requires additional training and experience to fill the gaps in skills and tools. It is recommended to consider the candidate for a junior or associate Data Scientist role, or to provide additional training and mentorship to help the candidate grow into the full Data Scientist role.

Candidate: Weak Candidate
Score: 0
Fit Level: Weak
Recommendation: Not recommended for the Data Scientist role due to significant gaps in required skills, tools, and experience. The candidate may be considered for an entry-level role in a different field or for a training program to develop the necessary skills for the Data Scientist position.


debugging example

In [19]:
bad_extraction_prompt = PromptTemplate(
    template="""
Extract candidate skills, tools, and experience from this resume.
You may infer likely skills if the profile suggests them.

Resume:
{resume_text}
""",
    input_variables=["resume_text"]
)

bad_extraction_chain = bad_extraction_prompt | llm | text_parser

In [20]:
bad_output = bad_extraction_chain.invoke({"resume_text": weak_resume})
print(bad_output)

Based on the provided resume, the following candidate skills, tools, and experience can be extracted:

**Skills:**
1. MS Office
2. Communication
3. HTML
4. CSS
5. Presentation (likely skill, inferred from college presentation projects)
6. Teamwork/Collaboration (likely skill, inferred from participation in coding club)
7. Basic coding skills (likely skill, inferred from participation in coding club)

**Tools:**
1. Word
2. PowerPoint
3. Web development tools (likely tool, inferred from HTML and CSS skills)

**Experience:**
1. College presentation projects (academic project experience)
2. Participation in coding club (extracurricular experience)
3. No direct industry experience (fresher)

Note: As the candidate is a fresher, the experience section is limited to academic and extracurricular activities. The skills and tools listed are based on the provided information and some inferences made from the candidate's background.


In [21]:
fixed_output = extraction_chain.invoke({"resume_text": weak_resume})
print(json.dumps(fixed_output, indent=2))

{
  "skills": [
    "MS Office",
    "Communication",
    "HTML",
    "CSS"
  ],
  "tools": [
    "Word",
    "PowerPoint"
  ],
  "experience_summary": "Fresher",
  "years_of_experience": "Not explicitly mentioned"
}


add LangSmith tags

In [22]:
def evaluate_resume_with_tags(resume_text: str, job_description: str, candidate_name: str) -> Dict[str, Any]:
    extracted = extraction_chain.with_config(
        {"run_name": "Skill Extraction", "tags": [candidate_name, "extraction"]}
    ).invoke({"resume_text": resume_text})

    matched = matching_chain.with_config(
        {"run_name": "Matching Logic", "tags": [candidate_name, "matching"]}
    ).invoke({
        "job_description": job_description,
        "extracted_data": json.dumps(extracted, indent=2)
    })

    scored = scoring_chain.with_config(
        {"run_name": "Scoring", "tags": [candidate_name, "scoring"]}
    ).invoke({
        "match_data": json.dumps(matched, indent=2)
    })

    explained = explanation_chain.with_config(
        {"run_name": "Explanation", "tags": [candidate_name, "explanation"]}
    ).invoke({
        "job_description": job_description,
        "resume_text": resume_text,
        "match_data": json.dumps(matched, indent=2),
        "score_data": json.dumps(scored, indent=2)
    })

    return {
        "candidate_name": candidate_name,
        "extraction": extracted,
        "match": matched,
        "score": scored,
        "explanation": explained
    }

In [23]:
strong_result = evaluate_resume_with_tags(strong_resume, job_description, "Strong")
average_result = evaluate_resume_with_tags(average_resume, job_description, "Average")
weak_result = evaluate_resume_with_tags(weak_resume, job_description, "Weak")

In [24]:
print_result(strong_result)
print_result(average_result)
print_result(weak_result)

Candidate: Strong
--------------------------------------------------------------------------------
EXTRACTION:
{
  "skills": [
    "Python",
    "Machine Learning",
    "Statistics",
    "SQL",
    "Data Visualization",
    "Feature Engineering",
    "Model Evaluation"
  ],
  "tools": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "Jupyter Notebook",
    "Git",
    "Matplotlib"
  ],
  "experience_summary": "Data Scientist at ABC Analytics",
  "years_of_experience": "3 years"
}
--------------------------------------------------------------------------------
MATCH:
{
  "matched_skills": [
    "Python",
    "Machine Learning",
    "Statistics",
    "SQL",
    "Data Visualization",
    "Model Evaluation"
  ],
  "missing_skills": [],
  "matched_tools": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "Jupyter Notebook",
    "Git"
  ],
  "missing_tools": [],
  "experience_match": "Matched",
  "fit_level": "Strong"
}
------------------------------

In [25]:
def print_result(result):
    print("="*80)
    print("Candidate:", result["candidate_name"])
    print("-"*80)
    print("EXTRACTION:")
    print(json.dumps(result["extraction"], indent=2))
    print("-"*80)
    print("MATCH:")
    print(json.dumps(result["match"], indent=2))
    print("-"*80)
    print("SCORE:")
    print(json.dumps(result["score"], indent=2))
    print("-"*80)
    print("EXPLANATION:")
    print(json.dumps(result["explanation"], indent=2))
    print("="*80)

In [27]:
print_result(strong_result)
print_result(average_result)
print_result(weak_result)

Candidate: Strong
--------------------------------------------------------------------------------
EXTRACTION:
{
  "skills": [
    "Python",
    "Machine Learning",
    "Statistics",
    "SQL",
    "Data Visualization",
    "Feature Engineering",
    "Model Evaluation"
  ],
  "tools": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "Jupyter Notebook",
    "Git",
    "Matplotlib"
  ],
  "experience_summary": "Data Scientist at ABC Analytics",
  "years_of_experience": "3 years"
}
--------------------------------------------------------------------------------
MATCH:
{
  "matched_skills": [
    "Python",
    "Machine Learning",
    "Statistics",
    "SQL",
    "Data Visualization",
    "Model Evaluation"
  ],
  "missing_skills": [],
  "matched_tools": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "Jupyter Notebook",
    "Git"
  ],
  "missing_tools": [],
  "experience_match": "Matched",
  "fit_level": "Strong"
}
------------------------------

In [ ]:
d